In [23]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'
import keras
import kagglehub
import pandas as pd

In [15]:
path = kagglehub.dataset_download("jangedoo/utkface-new")

Using Colab cache for faster access to the 'utkface-new' dataset.


In [ ]:
base_folder = os.path.join(path, 'utkface_aligned_cropped', 'UTKFace')

# Fallback: If that subfolder doesn't exist, use the base path
if not os.path.exists(base_folder):
    base_folder = path

age = []
gender = []
image_path = []

# 3. Loop through the correct directory
for file in os.listdir(base_folder):
    if file.endswith(".jpg"):
        try:
            parts = file.split('_')
            age.append(int(parts[0]))
            gender.append(int(parts[1]))
            # Store the full path so you can load the image later
            image_path.append(os.path.join(base_folder, file))
        except (ValueError, IndexError):
            continue

print(f"Successfully loaded {len(age)} images.")

Using Colab cache for faster access to the 'utkface-new' dataset.
Dataset downloaded to: /kaggle/input/utkface-new
Successfully loaded 23708 images.


In [24]:
df = pd.DataFrame({
    'age':age,
    'gender':gender,
    'image_path':image_path
})

In [25]:
df.head()

,age,gender,image_path
0,26,0,/kaggle/input/utkface-new/utkface_aligned_crop...
1,22,1,/kaggle/input/utkface-new/utkface_aligned_crop...
2,21,1,/kaggle/input/utkface-new/utkface_aligned_crop...
3,28,0,/kaggle/input/utkface-new/utkface_aligned_crop...
4,17,1,/kaggle/input/utkface-new/utkface_aligned_crop...


In [26]:
train_df = df.sample(frac=1, random_state=42).iloc[:20000]
test_df = df.sample(frac=1, random_state=42).iloc[20000]

In [29]:
train_df.shape

(20000, 3)

In [28]:
test_df.shape

(3,)

In [ ]:
data_augementation = keras.Sequential(
    [
        keras.layers.Rescaling(1./255),
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(30),
        keras.layers.RandomZoom(0.1),
        keras.layers.RandomShear(0.2),

    ]
)

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')